# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Key fields for my lane (Content Refresh Prioritization), described before any testing.
All the traffic-like columns are heavy-tailed — a few huge pages, a long tail of small ones —
so I use medians and percentiles below, not just means, and I'll bucket by tier rather than
run raw Pearson correlation on these later.

| Field | n | mean | median | p90 | max |
|---|---|---|---|---|---|
| `impressions_90d` | 30,000 | 5,200.4 | 731.0 | 12,136.4 | 517,715 |
| `clicks_90d` | 30,000 | 16.1 | 1.0 | 32.0 | 4,178 |
| `ctr` | 30,000 | 0.51 | 0.07 | 0.65 | 100.0 |
| `days_since_last_update` | 30,000 | 46.1 | 20.0 | 104.0 | 373 |
| `search_volume` | 27,532 (2,468 blank) | 158.9 | 10.0 | 110.0 | 74,000 |

The gap between mean and median on `impressions_90d` (mean 5,200 vs median 731) and
`search_volume` (mean 159 vs median 10) is the heavy tail showing itself directly: a handful
of very large pages/keywords pull the mean far above where most rows actually sit. `ctr`'s max
of 100.0 isn't an error — it's the "0 clicks isn't possible with >0 impressions but 1 click on
1 impression = 100%" edge case at the bottom of the volume distribution, which is exactly why
the CTR-vs-position test below needs a volume floor.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

fields = ["impressions_90d", "clicks_90d", "ctr", "days_since_last_update", "search_volume"]
desc = df[fields].describe(percentiles=[0.5, 0.9]).T
desc["n_non_null"] = df[fields].count()
print(desc[["n_non_null", "mean", "50%", "90%", "max"]].round(2))

                        n_non_null     mean     50%       90%       max
impressions_90d              30000  5200.37  731.00  12136.40  517715.0
clicks_90d                   30000    16.10    1.00     32.00    4178.0
ctr                          30000     0.51    0.07      0.65     100.0
days_since_last_update       30000    46.10   20.00    104.00     373.0
search_volume                27532   158.88   10.00    110.00   74000.0


## 2. Signal test #1 / #2 / #3 (verdict each)

### Test 1 — "Stale content declines more" (behind the refresh flags)

Claim: pages that haven't been updated in a while are more likely to be declining.
Bucket: `freshness_tier` (from `days_since_last_update`) vs `is_declining_label`. Base rate
across all 30,000 rows: **54.2%**.

| freshness_tier | n | decline_rate |
|---|---|---|
| 0-30 | 20,480 | 51.1% |
| 31-90 | 175 | 58.9% |
| 91-180 | 9,171 | **61.1%** |
| 181+ | 174 | 47.1% |

**Verdict: MIXED.** The two buckets with real n show the expected direction — `91-180`
(n=9,171) sits at 61.1%, above the `0-30` fresh bucket (51.1%, n=20,480) and the 54.2% base
rate. But `181+` (n=174) drops back to 47.1%, below baseline, so the relationship isn't a
straight line. I trust the `91-180` reading and don't trust `181+` either way (small n).

### Test 2 — "Better position gets more clicks" (behind the CTR-fix logic)

Claim: pages ranking higher get a higher click-through rate. Raw bucket (no floor) is
misleading — `top_3`'s median CTR reads as 0.00 because its median `impressions_90d` is only
**3** (76.8% of `top_3` rows have zero clicks: near-zero-volume queries, not real winners).
Refiltered to `impressions_90d >= 300` (a real visibility floor):

| position_tier | n | median_ctr |
|---|---|---|
| page_1 | 7,623 | 0.23 |
| top_3 | 485 | 0.20 |
| striking | 5,078 | 0.17 |
| page_3_5 | 5,004 | 0.08 |
| deep | 562 | 0.00 |

**Verdict: CONFIRMED.** Once low-volume noise is filtered, CTR drops cleanly as position
worsens. This only became visible after applying the data dictionary's own volume-floor
warning — the naive version of this test would have said the opposite.

### Test 3 — "Higher keyword search volume means more actual traffic" (behind quick-win sizing)

Claim: a page targeting a keyword with a bigger search-volume estimate should be pulling in
more real impressions — the basic assumption behind ranking "quick win" candidates by keyword
volume. Bucket: `search_volume` (manual tiers, since the raw values are extremely skewed —
median 10, mean 159) vs `impressions_90d`.

| search_volume tier | n | median impressions_90d | mean impressions_90d |
|---|---|---|---|
| 0-9 | 11,081 | 998.0 | 5,919.1 |
| 10-50 | 12,300 | 877.0 | 5,261.3 |
| 51-200 | 2,081 | 843.0 | 6,270.4 |
| 200+ | 2,070 | 789.5 | 5,517.5 |

Spearman rank correlation between `search_volume` and `impressions_90d`: **ρ = -0.029**
(statistically significant only because n is huge — practically flat).

**Verdict: FALSE.** Median impressions barely move across tiers, and the correlation is
effectively zero. In this slice, a keyword's estimated search volume does not predict how much
real traffic a page actually pulls in — these are already-published, already-ranking pages, so
their traffic looks driven by their current position/content quality, not the keyword's
theoretical ceiling. Sizing "quick wins" by `search_volume` alone would be sizing them by a
number that isn't actually connected to the outcome, in this dataset.

In [2]:
from scipy.stats import spearmanr

# --- Test 1: staleness ---
sig1 = df.groupby("freshness_tier").agg(n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean"))
sig1["decline_rate_pct"] = (sig1["decline_rate"] * 100).round(1)
print("Test 1 — freshness_tier vs decline rate (base rate {:.1f}%)".format(df["is_declining_label"].mean()*100))
print(sig1[["n", "decline_rate_pct"]])
print()

# --- Test 2: CTR vs position, volume-floored ---
floor = df[df["impressions_90d"] >= 300]
sig2 = floor.groupby("position_tier").agg(n=("ctr", "size"), median_ctr=("ctr", "median")).round(3)
print("Test 2 — position_tier vs median CTR (impressions_90d >= 300 floor)")
print(sig2)
print()

# --- Test 3: search_volume vs impressions_90d ---
sv = df.dropna(subset=["search_volume"]).copy()
bins = [-1, 9, 50, 200, 10**9]
labels = ["0-9", "10-50", "51-200", "200+"]
sv["sv_tier"] = pd.cut(sv["search_volume"], bins=bins, labels=labels)
sig3 = sv.groupby("sv_tier", observed=True).agg(
    n=("impressions_90d", "size"), median_impr=("impressions_90d", "median"), mean_impr=("impressions_90d", "mean")
).round(1)
print("Test 3 — search_volume tier vs impressions_90d")
print(sig3)
rho, p = spearmanr(sv["search_volume"], sv["impressions_90d"])
print(f"Spearman rho: {rho:.3f}, p={p:.2e}")

Test 1 — freshness_tier vs decline rate (base rate 54.2%)
                    n  decline_rate_pct
freshness_tier                         
0-30            20480              51.1
181+              174              47.1
31-90             175              58.9
91-180           9171              61.1

Test 2 — position_tier vs median CTR (impressions_90d >= 300 floor)
                  n  median_ctr
position_tier                  
deep            562        0.00
page_1         7623        0.23
page_3_5       5004        0.08
striking       5078        0.17
top_3           485        0.20



Test 3 — search_volume tier vs impressions_90d
             n  median_impr  mean_impr
sv_tier                               
0-9      11081        998.0     5919.1
10-50    12300        877.0     5261.3
51-200    2081        843.0     6270.4
200+      2070        789.5     5517.5
Spearman rho: -0.029, p=1.40e-06


## 3. The flag-linked test

I'm going deeper on Test 2 (CTR vs position), since it's the one directly behind FlyRank's
CTR-fix flag. Confirming "better position → higher CTR" is necessary but not sufficient for
that flag to matter — the flag's real assumption is that a page whose CTR falls **short of**
what its own position tier normally gets is a page worth prioritizing, i.e. that gap should
predict something, not just describe something.

So the actual test: among visible pages (`impressions_90d >= 300`), does `ctr_underperforming`
(a page's own CTR below its tier's median CTR) predict a higher decline rate?

| ctr_underperforming | n | decline_rate |
|---|---|---|
| False (CTR at/above tier expectation) | 9,968 | 54.0% |
| **True (CTR below tier expectation)** | 8,784 | **65.7%** |

Base rate on this visible slice: 59.5%.

**Verdict: CONFIRMED.** Pages flagged as CTR-underperforming for their position are noticeably
more likely to be declining (65.7% vs 54.0%, both buckets well above the sample-size floor).
This is the assumption the CTR-fix flag actually needs to hold — not just "position predicts
CTR" (Test 2), but "falling short of that prediction predicts trouble." It does, in this
slice.

In [3]:
floor = df[df["impressions_90d"] >= 300].copy()
expected_ctr_by_tier = floor.groupby("position_tier")["ctr"].median()
floor["expected_ctr"] = floor["position_tier"].map(expected_ctr_by_tier)
floor["ctr_underperforming"] = floor["ctr"] < floor["expected_ctr"]

flag_test = floor.groupby("ctr_underperforming").agg(
    n=("is_declining_label", "size"), decline_rate=("is_declining_label", "mean")
)
flag_test["decline_rate_pct"] = (flag_test["decline_rate"] * 100).round(1)
print(flag_test[["n", "decline_rate_pct"]])
print("Base rate on this visible slice:", round(floor["is_declining_label"].mean() * 100, 1), "%")

                        n  decline_rate_pct
ctr_underperforming                        
False                9968              54.0
True                 8784              65.7
Base rate on this visible slice: 59.5 %


## 4. What this means in practice

A content team should trust the CTR-fix flag most: pages underperforming their position's
typical CTR really are more likely to be declining (65.7% vs 54.0%), so that gap is worth
acting on. The staleness flag deserves a narrower rule than "anything old is at risk" — only
the 91–180-day window showed real elevated risk here, and the oldest content did not follow
the same pattern. And "quick win" candidates should not be sized by keyword search-volume
estimates alone — in this data that number was essentially uncorrelated with the traffic a
page actually receives, so a size-by-search-volume queue would likely misorder itself.

In [4]:
summary = {
    "staleness_vs_decline": "MIXED — confirmed for 91-180d window (n=9,171), reverses at 181+ (n=174)",
    "ctr_vs_position": "CONFIRMED — clean drop once impressions_90d >= 300 floor applied",
    "search_volume_vs_impressions": "FALSE — Spearman rho -0.029, medians flat across tiers",
    "ctr_underperforming_vs_decline (flag-linked)": "CONFIRMED — 65.7% vs 54.0% decline rate",
}
for k, v in summary.items():
    print(f"- {k}: {v}")

- staleness_vs_decline: MIXED — confirmed for 91-180d window (n=9,171), reverses at 181+ (n=174)
- ctr_vs_position: CONFIRMED — clean drop once impressions_90d >= 300 floor applied
- search_volume_vs_impressions: FALSE — Spearman rho -0.029, medians flat across tiers
- ctr_underperforming_vs_decline (flag-linked): CONFIRMED — 65.7% vs 54.0% decline rate


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.